# 04 - Hill climbing loop: melhorando um plano de venda de milho

Hill climbing é uma busca local. Começa de uma solução, olha os **vizinhos** (pequenas mudanças dela), escolhe o melhor vizinho e segue para ele se for melhor que a solução atual. O loop para quando nenhum vizinho melhora: isso é um **ótimo local**.

Fluxo:

```
plano inicial → avalia → gera vizinhos → avalia os vizinhos → melhor vizinho é melhor que o atual?
                                                              → sim: anda para ele e repete
                                                              → não: para (ótimo local)
```

Em loop engineering, o hill climbing é um loop de melhoria. Ele tem três partes que precisam ser projetadas:

1. **Avaliação:** uma função de score que diz o quão bom é um plano.
2. **Vizinhança:** como gerar pequenas mudanças do plano atual.
3. **Critério de parada:** parar quando não há melhora, ou quando acaba o orçamento de iterações.

Neste notebook, o problema é um produtor do Paraná que decide quanto milho vender agora, em 30 dias e em 60 dias. A produção é fictícia, e o score é a receita esperada menos um prêmio pelo risco.

Em duas partes:

1. **Hill climbing determinístico:** a vizinhança é gerada por regra, sem LLM.
2. **Hill climbing com LLM:** o modelo propõe os vizinhos, e a função de avaliação decide quais aceitar.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

O rastreamento usa o MLflow com um banco SQLite, `mlflow.db`, criado na pasta deste notebook. A versão com LLM vira um trace.

In [1]:
import json
import math
import os

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

import mlflow

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "hill-climbing-milho"
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.langchain.autolog()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

2026/11/17 20:03:43 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/leonardo/Documentos/github/loop_engineering_study/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 2. Dados simulados e ferramentas

Mesmos dados do notebook 02. Usamos a ferramenta de preço para pegar o preço de hoje em Cascavel.

In [2]:
PRECOS_SACA = {
    "paranagua": {"praca": "Paranaguá (PR)", "preco_rs_saca": 118.50, "variacao_7_dias_pct": 1.8},
    "cascavel": {"praca": "Cascavel (PR)", "preco_rs_saca": 112.30, "variacao_7_dias_pct": 2.4},
    "rio verde": {"praca": "Rio Verde (GO)", "preco_rs_saca": 108.90, "variacao_7_dias_pct": 0.6},
    "campinas": {"praca": "Campinas (SP)", "preco_rs_saca": 120.10, "variacao_7_dias_pct": -0.9},
}

CLIMA = {
    "parana": {"regiao": "Oeste do Paraná", "chuva_ultimos_7_dias_mm": 12, "chuva_prevista_15_dias_mm": 18,
               "risco": "baixo", "observacao": "Chuva irregular no fim do mês, sem sinal de seca forte."},
    "mato grosso": {"regiao": "Mato Grosso", "chuva_ultimos_7_dias_mm": 4, "chuva_prevista_15_dias_mm": 6,
                    "risco": "alto", "observacao": "Veranico prolongado, risco de perda na segunda safra."},
    "goias": {"regiao": "Goiás", "chuva_ultimos_7_dias_mm": 7, "chuva_prevista_15_dias_mm": 10,
              "risco": "médio", "observacao": "Chuva abaixo da média para a época."},
}

ESTOQUES = {
    "safra_estimada_mil_t": 128_000,
    "consumo_estimado_mil_t": 79_000,
    "estoque_inicial_mil_t": 21_000,
    "exportacao_estimada_mil_t": 42_000,
    "estoque_final_estimado_mil_t": 28_000,
    "relacao_estoque_consumo_pct": 35.4,
    "leitura": "Estoque final abaixo da média dos últimos 5 anos.",
}


@tool
def consultar_preco_milho(praca: str) -> str:
    """Consulta o preço spot do milho em uma praça. Praças válidas: paranagua, cascavel, rio verde, campinas."""
    chave = praca.lower().strip()
    if chave not in PRECOS_SACA:
        return json.dumps({"erro": f"Praça '{praca}' não encontrada.", "pracas_validas": list(PRECOS_SACA)}, ensure_ascii=False)
    return json.dumps(PRECOS_SACA[chave], ensure_ascii=False)


@tool
def consultar_clima(regiao: str) -> str:
    """Consulta a chuva recente, a previsão e o risco climático de uma região produtora. Regiões válidas: parana, mato grosso, goias."""
    chave = regiao.lower().strip()
    if chave not in CLIMA:
        return json.dumps({"erro": f"Região '{regiao}' não encontrada.", "regioes_validas": list(CLIMA)}, ensure_ascii=False)
    return json.dumps(CLIMA[chave], ensure_ascii=False)


@tool
def consultar_estoques() -> str:
    """Consulta o balanço de oferta e demanda do milho no Brasil: safra, consumo, exportação e estoque final."""
    return json.dumps(ESTOQUES, ensure_ascii=False)


FERRAMENTAS = [consultar_preco_milho, consultar_clima, consultar_estoques]

## 3. Função de avaliação

O plano é uma tupla `(a, b)` em porcentagem da produção:

- `a`: fração vendida **agora**;
- `b`: fração vendida **em 30 dias**;
- `c = 100 - a - b`: fração vendida **em 60 dias**.

Como `c` é o que sobra, todo plano com `a ≥ 0`, `b ≥ 0` e `a + b ≤ 100` é válido.

Os preços futuros vêm de três cenários fictícios, com probabilidade. Guardar milho tem custo de armazenagem, por saca e por mês.

O score é a receita esperada menos um prêmio pelo risco, `média - λ × desvio padrão`, e menos uma **taxa fixa de R$ 8.000 por tranche usada** (contrato, frete e análise de cada venda). Uma tranche é usada quando a fração correspondente é maior que zero.

A taxa fixa é o que deixa o problema interessante. Ela faz o score cair de repente quando uma tranche entra ou sai, e isso cria **ótimos locais**: planos que nenhum vizinho de 5 pontos consegue melhorar, mesmo sem ser o melhor plano possível.

In [3]:
PRODUCAO_SACAS = 10_000
CUSTO_ARMAZENAGEM_POR_MES = 1.5  # R$ por saca por mês
CUSTO_FIXO_POR_TRANCHE = 8_000  # R$ por tranche usada (a, b ou c maior que zero)
LAMBDA_RISCO = 0.1
PRECO_HOJE = json.loads(consultar_preco_milho.invoke({"praca": "cascavel"}))["preco_rs_saca"]

CENARIOS = [
    {"nome": "baixa", "prob": 0.3, "mult_30d": 0.95, "mult_60d": 0.80},
    {"nome": "base", "prob": 0.4, "mult_30d": 1.03, "mult_60d": 1.04},
    {"nome": "alta", "prob": 0.3, "mult_30d": 1.10, "mult_60d": 1.22},
]


def factivel(a: int, b: int) -> bool:
    return a >= 0 and b >= 0 and a + b <= 100


def receita_no_cenario(a: int, b: int, cenario: dict) -> float:
    c = 100 - a - b
    preco_30d = PRECO_HOJE * cenario["mult_30d"]
    preco_60d = PRECO_HOJE * cenario["mult_60d"]
    return PRODUCAO_SACAS * (
        a / 100 * PRECO_HOJE
        + b / 100 * (preco_30d - 1 * CUSTO_ARMAZENAGEM_POR_MES)
        + c / 100 * (preco_60d - 2 * CUSTO_ARMAZENAGEM_POR_MES)
    )


def tranches_usadas(a: int, b: int) -> int:
    c = 100 - a - b
    return (a > 0) + (b > 0) + (c > 0)


def avaliar(a: int, b: int) -> float:
    receitas = [receita_no_cenario(a, b, cen) for cen in CENARIOS]
    media = sum(cen["prob"] * r for cen, r in zip(CENARIOS, receitas))
    variancia = sum(cen["prob"] * (r - media) ** 2 for cen, r in zip(CENARIOS, receitas))
    return media - LAMBDA_RISCO * math.sqrt(variancia) - CUSTO_FIXO_POR_TRANCHE * tranches_usadas(a, b)


print(f"Preço de hoje em Cascavel: R$ {PRECO_HOJE:.2f}/saca")
print(f"Score de vender tudo agora (a=100, b=0): R$ {avaliar(100, 0):,.0f}")
print(f"Score de vender tudo em 30 dias (a=0, b=100): R$ {avaliar(0, 100):,.0f}")
print(f"Score de dividir em partes iguais (a=34, b=33): R$ {avaliar(34, 33):,.0f}")

Preço de hoje em Cascavel: R$ 112.30/saca
Score de vender tudo agora (a=100, b=0): R$ 1,115,000
Score de vender tudo em 30 dias (a=0, b=100): R$ 1,123,791
Score de dividir em partes iguais (a=34, b=33): R$ 1,094,103


## 4. Referência: busca em grade

Antes do hill climbing, uma busca exaustiva testa todos os planos em passos de 5 pontos percentuais. Com 231 planos, é barato, então ela serve de referência para checar o resultado do loop. Ela não tem ótimo local: olha todos os planos.

Em problemas grandes, a busca exaustiva não é viável. É aí que o hill climbing faz sentido.

In [4]:
PASSO = 5

grade = [(a, b) for a in range(0, 101, PASSO) for b in range(0, 101 - a, PASSO)]
melhor_grade = max(grade, key=lambda p: avaliar(*p))

print(f"Planos testados na grade: {len(grade)}")
print(f"Melhor plano na grade: a={melhor_grade[0]}%, b={melhor_grade[1]}%, c={100 - sum(melhor_grade)}%")
print(f"Score: R$ {avaliar(*melhor_grade):,.0f}")

Planos testados na grade: 231
Melhor plano na grade: a=0%, b=100%, c=0%
Score: R$ 1,123,791


## 5. Hill climbing determinístico

A vizinhança é regra fixa: de cada plano, dá para mover 5 pontos para `a` ou para `b`, para cima ou para baixo. A parte que sobra vai para `c` ou sai de `c`, porque `c` é sempre o resto. Ficam só os vizinhos factíveis.

A cada iteração, o loop escolhe o melhor vizinho e só anda se ele for estritamente melhor. Quando não há melhora, o loop para. Esse é o hill climbing de subida mais íngreme.

Rodamos a partir de cinco pontos de partida. Como a taxa fixa cria ótimos locais, pontos de partida diferentes podem parar em planos diferentes. A grade mostra qual é o melhor de verdade.

In [5]:
MOVIMENTOS = [(PASSO, 0), (-PASSO, 0), (0, PASSO), (0, -PASSO)]


def vizinhos(a: int, b: int) -> list[tuple[int, int]]:
    return [(a + da, b + db) for da, db in MOVIMENTOS if factivel(a + da, b + db)]


def hill_climbing(inicio: tuple[int, int], max_iter: int = 50) -> tuple[tuple[int, int], float, list]:
    atual = inicio
    score = avaliar(*atual)
    historico = [(0, atual, score)]

    for iteracao in range(1, max_iter + 1):
        melhor = max(vizinhos(*atual), key=lambda p: avaliar(*p))
        if avaliar(*melhor) <= score:
            break
        atual = melhor
        score = avaliar(*atual)
        historico.append((iteracao, atual, score))

    return atual, score, historico


for inicio in [(0, 0), (100, 0), (50, 25), (0, 100), (34, 33)]:
    final, score_final, historico = hill_climbing(inicio)
    print(f"\nInício a={inicio[0]}%, b={inicio[1]}%:")
    for it, (a, b), score in historico:
        print(f"  iteração {it:>2}: a={a:>3}%, b={b:>3}%, c={100 - a - b:>3}%, score R$ {score:,.0f}")
    print(f"  parou em a={final[0]}%, b={final[1]}%, c={100 - sum(final)}%, score R$ {score_final:,.0f}")


Início a=0%, b=0%:
  iteração  0: a=  0%, b=  0%, c=100%, score R$ 1,091,364
  parou em a=0%, b=0%, c=100%, score R$ 1,091,364

Início a=100%, b=0%:
  iteração  0: a=100%, b=  0%, c=  0%, score R$ 1,115,000
  parou em a=100%, b=0%, c=0%, score R$ 1,115,000

Início a=50%, b=25%:
  iteração  0: a= 50%, b= 25%, c= 25%, score R$ 1,095,290
  iteração  1: a= 50%, b= 30%, c= 20%, score R$ 1,096,912
  iteração  2: a= 50%, b= 35%, c= 15%, score R$ 1,098,533
  iteração  3: a= 50%, b= 40%, c= 10%, score R$ 1,100,154
  iteração  4: a= 50%, b= 45%, c=  5%, score R$ 1,101,775
  iteração  5: a= 50%, b= 50%, c=  0%, score R$ 1,111,396
  parou em a=50%, b=50%, c=0%, score R$ 1,111,396

Início a=0%, b=100%:
  iteração  0: a=  0%, b=100%, c=  0%, score R$ 1,123,791
  parou em a=0%, b=100%, c=0%, score R$ 1,123,791

Início a=34%, b=33%:
  iteração  0: a= 34%, b= 33%, c= 33%, score R$ 1,094,103
  iteração  1: a= 34%, b= 38%, c= 28%, score R$ 1,095,725
  iteração  2: a= 34%, b= 43%, c= 23%, score R$ 1,097,

## 6. Hill climbing com LLM

Aqui o modelo propõe os vizinhos, em vez de uma regra fixa. Ele recebe o plano atual, o score e as regras, e devolve três planos com justificativa. Isso é o papel do agente no loop.

A função de avaliação continua sendo a mesma e decide tudo. Se o modelo propuser um plano que não é vizinho ou não é factível, ele é descartado. O modelo não aceita o próprio plano: ele só sugere, e o score decide.

O loop para no mesmo critério: quando nenhum plano proposto melhora o score, ou quando acaba o orçamento de iterações. Começamos do plano de 34% agora, 33% em 30 dias e 33% em 60 dias, o mesmo ponto de partida da versão determinística.

In [6]:
class Proposta(BaseModel):
    vender_agora_pct: int = Field(ge=0, le=100, description="Porcentagem da produção vendida agora.")
    vender_30d_pct: int = Field(ge=0, le=100, description="Porcentagem da produção vendida em 30 dias.")
    justificativa: str = Field(description="Por que este plano pode ser melhor que o atual, em uma frase.")


class Propostas(BaseModel):
    propostas: list[Proposta] = Field(description="Três planos vizinhos ao plano atual.")


proponente = llm.with_structured_output(Propostas)
VIZINHANCA_MAX_PONTOS = 20  # distância máxima, em pontos percentuais, entre o plano atual e um vizinho


@mlflow.trace(name="hill_climbing_llm")
def hill_climbing_llm(inicio: tuple[int, int], max_iter: int = 6) -> dict:
    atual = inicio
    score = avaliar(*atual)
    historico = [(0, atual, score)]

    for iteracao in range(1, max_iter + 1):
        a, b = atual
        resposta = proponente.invoke([
            SystemMessage(content=(
                "Você otimiza um plano de venda de milho. O plano tem três partes: vender agora (a), "
                "vender em 30 dias (b) e vender em 60 dias (c = 100 - a - b). "
                "Guardar milho custa armazenagem. O score é a receita esperada menos 0,1 vez o desvio padrão "
                f"menos R$ {CUSTO_FIXO_POR_TRANCHE:,} por tranche usada (uma tranche é usada quando a fração é maior que zero). "
                f"Preço de hoje: R$ {PRECO_HOJE:.2f} por saca. Cenários: "
                + "; ".join(f"{c['nome']} (prob. {c['prob']}, preço em 30 dias x{c['mult_30d']}, em 60 dias x{c['mult_60d']})" for c in CENARIOS)
                + ". Proponha planos vizinhos, que mudem o plano atual em poucos pontos percentuais."
            )),
            HumanMessage(content=(
                f"Plano atual: vender agora {a}%, em 30 dias {b}%, em 60 dias {100 - a - b}%. "
                f"Score atual: R$ {score:,.0f}. Proponha três planos vizinhos com justificativa."
            )),
        ])

        candidatos = [
            (p.vender_agora_pct, p.vender_30d_pct, p.justificativa)
            for p in resposta.propostas
            if factivel(p.vender_agora_pct, p.vender_30d_pct)
            and abs(p.vender_agora_pct - a) + abs(p.vender_30d_pct - b) <= VIZINHANCA_MAX_PONTOS
        ]
        print(f"iteração {iteracao}: {len(resposta.propostas)} propostas, {len(candidatos)} válidas")
        if not candidatos:
            print("  nenhuma proposta válida: parada")
            break

        melhor_a, melhor_b, justificativa = max(candidatos, key=lambda x: avaliar(x[0], x[1]))
        score_melhor = avaliar(melhor_a, melhor_b)
        if score_melhor <= score:
            print(f"  nenhuma proposta melhora o score (melhor: R$ {score_melhor:,.0f}): parada")
            break

        print(f"  aceita a={melhor_a}%, b={melhor_b}%, score R$ {score_melhor:,.0f}")
        print(f"  justificativa: {justificativa}")
        atual = (melhor_a, melhor_b)
        score = score_melhor
        historico.append((iteracao, atual, score))

    return {"plano": atual, "score": score, "historico": historico, "trace_id": mlflow.get_active_trace_id()}


resultado_llm = hill_climbing_llm((34, 33), max_iter=8)

iteração 1: 3 propostas, 3 válidas
  aceita a=32%, b=35%, score R$ 1,094,279
  justificativa: Transferir parcela do imediato para 30 dias explora recuperação provável no curto prazo (cenário base/alta) com impacto reduzido em custos de armazenagem.


iteração 2: 3 propostas, 3 válidas
  aceita a=37%, b=33%, score R$ 1,094,812
  justificativa: Aumentar vendas imediatas para 37% reduz exposição a cenários de queda (x0.95/x0.8) e corta custo de armazenamento, baixando risco e possivelmente aumentando receita líquida.


iteração 3: 3 propostas, 3 válidas
  aceita a=37%, b=38%, score R$ 1,096,434
  justificativa: Desloca 5 pontos da venda em 60 dias para 30 dias para aproveitar leve maior preço esperado em 30 dias e reduzir exposição à volatilidade maior de 60 dias.


iteração 4: 3 propostas, 3 válidas
  aceita a=33%, b=45%, score R$ 1,097,758
  justificativa: Deslocar volume para 30 dias aproveita o cenário base/alta esperado de leve alta em 30 dias, mantendo diversificação entre tranches.


iteração 5: 3 propostas, 3 válidas
  aceita a=28%, b=50%, score R$ 1,098,198
  justificativa: Reduzir vendas imediatas em 5 p.p. e aumentar 30 dias captura o pequeno prêmio de preço esperado para 30 dias, elevando a receita esperada com um aumento modesto do risco.


iteração 6: 3 propostas, 3 válidas
  aceita a=30%, b=52%, score R$ 1,099,319
  justificativa: Pequena realocação do lote de 60 dias para agora e 30 dias para reduzir exposição à queda no horizonte longo, diminuindo o desvio padrão sem perder muito do upside.


iteração 7: 3 propostas, 3 válidas
  aceita a=30%, b=60%, score R$ 1,101,913
  justificativa: Movimentação mais acentuada para 30d (+8 p.p.), aproveitando seu maior multiplicador esperado e reduzindo a dependência do cenário de 60 dias.


iteração 8: 3 propostas, 3 válidas
  aceita a=30%, b=70%, score R$ 1,113,154
  justificativa: Elimina a tranche de 60 dias (10%) e transfere para 30 dias; 30d tem retorno esperado ligeiramente maior (~1,027x) e bem menor volatilidade que 60d, além de reduzir o custo por tranche em R$ 8.000.


## 7. Comparando os resultados

Os resultados precisam ser comparados com o ótimo da grade. Se o hill climbing chegou perto dele, o loop funcionou. Se ficou longe, o ponto de partida caiu num ótimo local, ou o loop parou cedo por falta de orçamento de iterações.

In [7]:
final_det, score_det, _ = hill_climbing((34, 33))  # mesmo ponto de partida da versão com LLM
plano_llm = resultado_llm["plano"]
score_llm = resultado_llm["score"]

print(f"Grade (referência):   a={melhor_grade[0]}%, b={melhor_grade[1]}%, score R$ {avaliar(*melhor_grade):,.0f}")
print(f"Determinístico:       a={final_det[0]}%, b={final_det[1]}%, score R$ {score_det:,.0f}")
print(f"Com LLM:              a={plano_llm[0]}%, b={plano_llm[1]}%, score R$ {score_llm:,.0f}")

Grade (referência):   a=0%, b=100%, score R$ 1,123,791
Determinístico:       a=34%, b=63%, score R$ 1,103,830
Com LLM:              a=30%, b=70%, score R$ 1,113,154


## 8. Acompanhando o loop no MLflow

A versão com LLM virou um trace no experimento `hill-climbing-milho`. Na árvore, cada iteração aparece como uma chamada ao modelo.

Para abrir a interface:

1. Abra um terminal na pasta `notebooks/05-LoopEngineering`.
2. Rode:

```bash
uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

3. Abra `http://127.0.0.1:5000`, escolha o experimento `hill-climbing-milho` e abra a aba **Traces**.
4. Pressione `Ctrl+C` para parar a interface.

A célula seguinte lê o trace pelo notebook.

In [8]:
mlflow.flush_trace_async_logging()  # garante que os traces já foram gravados

trace = mlflow.get_trace(resultado_llm["trace_id"])
chamadas_modelo = [s for s in trace.data.spans if s.span_type == "CHAT_MODEL"]
print(f"Trace do loop com LLM ({trace.info.trace_id})")
print(f"Chamadas ao modelo: {len(chamadas_modelo)}")

Trace do loop com LLM (tr-f79116dbda1af9dfa7614005e89809de)
Chamadas ao modelo: 8


## Observações

- Hill climbing não garante o ótimo global. Ele para no primeiro ótimo local. A taxa fixa por tranche cria degraus na função, e por isso pontos de partida diferentes param em planos diferentes.
- Comparar com a busca em grade mostra se o loop está funcionando. Sem essa referência, dá para aceitar um ótimo local como se fosse o melhor plano.
- Com LLM, o modelo é o gerador de candidatos e a função de avaliação é o juiz. Propostas ruins são descartadas, e o loop não depende de o modelo acertar.
- O loop precisa de limite de iterações e de um critério de parada claro: sem melhora, para. Sem limite, um proponente que sempre propõe algo "melhor" por pouco poderia rodar sem fim.